### State Level Model - How to select labeling method

In this notebook we intend to set up a standard for labeling each state by their main industry. Through previous investigation, we found that if we see real estate as a valid indsutry and label according to the dominant industry (GDP), then real estate will be the label for around 25 states, which is very unbalanced.

For the states where real estate is the leading industry, we are going to define a parameter alpha as a threshold for it's contribution in each state. Here are three ways to possibly define it:
1. the share of real estate in total GDP
2. the ratio of real estate and second leading industry in GDP
3. the difference of the share between real state and second leading industry in GDP

I will find the most reasonable definition of alpha and an optimal value under that definition by comparing the model performance for XGBoost (the best performing model for national level).

To start with, I'll incorporate the labeling parameter alpha into my old model, which labels each state with the industry with the biggest share.

Data source: https://apps.bea.gov/itable/?ReqID=70&step=1&_gl=1*19kveko*_ga*MTI3MzUwNTg4My4xNzczNTEzNzg2*_ga_J4698JNNFT*czE3NzM1MTM3ODYkbzEkZzEkdDE3NzM1MTcwNjgkajYwJGwwJGgw

Quarterly Gross Domestic Product (GDP) by State

SQGDP2 GDP by industry in current dollars

all areas, all statistics in the table, levels, 2024

In [1]:
# import data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r'E:\Erdos_DS_BootCamp\Project_my_work\State_labeling_data\SQGDP2_2024.csv', skiprows=3)

In [2]:
# Preprocessing

# remove regions that are not states
drop_codes_region = ['00000', '91000', '92000', '93000', '94000', '95000', '96000', '97000', '98000']
df = df[~df["GeoFIPS"].isin(drop_codes_region)].copy()

# convert quarters to numeric
quarter_cols = ["2024:Q1", "2024:Q2", "2024:Q3", "2024:Q4"]
df[quarter_cols] = df[quarter_cols].apply(pd.to_numeric, errors="coerce")

# compute annual GDP
df["GDP_2024"] = df[quarter_cols].sum(axis=1)

# subtract government (LineCode 83) from total GDP (LineCode 1) on the original df
gov = df[df["LineCode"] == 83][["GeoFIPS", "GDP_2024"] + quarter_cols]

tot = df[df["LineCode"] == 1].merge(gov, on="GeoFIPS", how="left", suffixes=("", "_gov"))

for c in quarter_cols + ["GDP_2024"]:
    df.loc[df["LineCode"] == 1, c] = (
        tot[c] - tot[f"{c}_gov"].fillna(0)
    ).values

# now remove supersets / subsets / government / addenda rows
drop_codes_industry = [2, 13, 25, 83, 84, 85, 86, 100, 101]
df = df[~df["LineCode"].isin(drop_codes_industry)].copy()

# separate adjusted total GDP
total_gdp = (
    df[df["LineCode"] == 1][["GeoName", "GDP_2024"]]
    .rename(columns={"GDP_2024": "TotalGDP"})
)

industries = df[df["LineCode"] != 1].copy()
industries = industries.merge(total_gdp, on="GeoName", how="left")

In [3]:
# Label each state by Location Quotient (LQ)
industries.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GeoFIPS      974 non-null    str    
 1   GeoName      969 non-null    str    
 2   LineCode     969 non-null    float64
 3   Description  969 non-null    str    
 4   2024:Q1      951 non-null    float64
 5   2024:Q2      951 non-null    float64
 6   2024:Q3      951 non-null    float64
 7   2024:Q4      951 non-null    float64
 8   GDP_2024     974 non-null    float64
 9   TotalGDP     969 non-null    float64
dtypes: float64(7), str(3)
memory usage: 125.0 KB


In [6]:
# state industry share
industries["state_share"] = industries["GDP_2024"] / industries["TotalGDP"]

# national totals by industry
national = (
    industries.groupby("Description", as_index=False)["GDP_2024"]
    .sum()
    .rename(columns={"GDP_2024": "NationalIndustryGDP"})
)

# national total GDP across all kept industries
national_total_gdp = national["NationalIndustryGDP"].sum()

# national industry share
national["national_share"] = national["NationalIndustryGDP"] / national_total_gdp

# merge national share back to each state-industry row
industries = industries.merge(
    national[["Description", "national_share"]],
    on="Description",
    how="left"
)

# location quotient
industries["LQ"] = industries["state_share"] / industries["national_share"]

# label each state by largest LQ
labels = industries.loc[
    industries.groupby("GeoName")["LQ"].idxmax(),
    ["GeoName", "Description", "LQ", "state_share", "national_share"]
].sort_values("GeoName")

In [5]:
industries["share"] = industries["GDP_2024"] / industries["TotalGDP"]
labels = industries.loc[
    industries.groupby("GeoName")["share"].idxmax(),
    ["GeoName","Description","share"]
]

In [6]:
label_groups = labels.groupby("Description")["GeoName"].apply(list)

for industry, states in label_groups.items():
    print(industry)
    print(states)
    print()

      Finance and insurance
['Delaware', 'Nebraska', 'New York', 'South Dakota']

      Information
['Washington']

      Manufacturing
['Alabama', 'Arkansas', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Michigan', 'Mississippi', 'Ohio', 'Wisconsin']

      Mining, quarrying, and oil and gas extraction
['New Mexico', 'North Dakota', 'West Virginia', 'Wyoming']

      Professional, scientific, and technical services
['District of Columbia', 'Massachusetts']

      Real estate and rental and leasing
['Arizona', 'California', 'Colorado', 'Connecticut', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Maine', 'Maryland', 'Minnesota', 'Missouri', 'Montana', 'Nevada', 'New Hampshire', 'New Jersey', 'North Carolina', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia']

      Transportation and warehousing
['Alaska']



In [7]:
labels["Description"].value_counts()

Description
Real estate and rental and leasing                  28
Manufacturing                                       11
Finance and insurance                                4
Mining, quarrying, and oil and gas extraction        4
Professional, scientific, and technical services     2
Transportation and warehousing                       1
Information                                          1
Name: count, dtype: int64